# Bag-of-words vektorer med dhlab

Hver tekst i subsettet blir tokenisert med `nb_tokenizer`.  
Vokabularet (=de unike tokensa) på tvers av alle tekstene utgjør dimensjonene i BoW-vektoren.

En tekst sin BoW-vektor vil ha verdien 0 for alle tokens som ikke er i teksten. 
For tokens som er i teksten, er verdien i hvert tokens dimensjon (antall forekomster/total frekvens i dhlaben). Dermed vil tokens som er frekvente uansett vektes ut, mens tokens som er sjeldne vil få høyere viktighet.


## Les og tokeniser subsettet

In [ ]:
from collections import Counter
from pathlib import Path

import pandas as pd
from nb_tokenizer import tokenize

SUBSET_DIR = Path("data/maalfrid/subset")
paths = sorted(SUBSET_DIR.glob("*/*.txt"))

texts = pd.Series(
    [path.read_text(encoding="utf-8") for path in paths],
    index=pd.Index([f"{path.parent.name}/{path.stem}" for path in paths]),
)
counts = {doc_id: Counter(tokenize(text)) for doc_id, text in texts.items()}
len(counts)

## Term-dokument matrise 
Hver rad i tabellen er et dokument  
Kolonnene er tokens.  
I hver celle er det antall ganger det tokenet forekommer i det dokumentet. 

In [ ]:
bow_counts = pd.DataFrame.from_dict(counts, orient="index").fillna(0)
bow_counts

In [ ]:
token_index = 23_000
token = bow_counts.columns[token_index]
print(
    f"Det er {len(bow_counts.columns)} unike tokens i subsettet. Tokenet på indeks {token_index} er '{token}'"
)
print(
    f"Dette tokenet forekommer {bow_counts[token].sum()} ganger totalt på tvers av alle tekstene"
)

## Finn de mest frekvente og minst frekvente orda i subsettet
Løp over og tell opp antall forekomster av hvert token.  
Sortér på frekvens

In [ ]:
all_tokens = bow_counts.columns
tokens_sums = pd.Series({token: bow_counts[token].sum() for token in all_tokens})
tokens_sums.sort_values(ascending=False).head(20)

In [ ]:
tokens_sums.sort_values(ascending=True).head(20)

## Skaler tokenfrekvensene med dhlab totals
Høyfrekvente ord på tvers av alle dokumenter er ikke spesielt informative.  
Vi skalerer alle ord basert på de 500 000 mest vanlige tokens i hele samlingen, med dhlab.totals

In [ ]:
from dhlab.api.dhlab_api import totals

TOP_WORDS = 500_000
freqs = totals(TOP_WORDS)["freq"]
min_freq = freqs.iloc[-1]

freqs.head(10)

In [ ]:
scaling = freqs.reindex(bow_counts.columns).fillna(min_freq)
bow = bow_counts / scaling

num_docs, vocab_size = bow.shape
print(
    f"Number of documents: {num_docs}\nNumber of unique tokens across documents: {vocab_size}"
)
print(
    f"{scaling.eq(min_freq).mean():.1%} of the vocabulary is outside the top {TOP_WORDS:,} words"
)

## Et eksempel på en skalert BoW-vector 

In [ ]:
first_vector = bow.iloc[0]

empty_dimensions = first_vector == 0
print(
    f"Vektoren har {empty_dimensions.sum()} dimensjoner som er 0 (-> tokenet som dimensjonen representerer finnes ikke i teksten)"
)
print(
    f"Det er {(~empty_dimensions).sum()} dimensjoner som ikke er 0, altså består teksten av {(~empty_dimensions).sum()} unike ord/tokens."
)


In [ ]:
# Vis de 20 tokensa høyest verdi i denne vektoren
first_vector.sort_values(ascending=False).head(20)

In [ ]:
# Vis de 20 tokensa lavest verdi i denne vektoren
first_vector[~empty_dimensions].sort_values().head(20)

## Flat ned til 2D-vektorer for visualisering


In [ ]:
from visualize import plot_by_domain, project_2d

projection = project_2d(bow.to_numpy(), bow.index)
projection.head()

## Scatterplot etter kildedomene

Hvert domene har sin egen farge og form.  
Klikk på et domene i forklaringen for å skjule eller vise det, og dobbeltklikk for å vise bare det domenet.  
Hold over en prikk for å se domenet og starten av teksten i panelet til høyre.

In [ ]:
plot_by_domain(projection, texts, "t-SNE of BOW embeddings")

## Mest og minst like domener
Hvert domene representeres ved sentroiden (gjennomsnitt av dokumententvektorene)   
Likhetsmålet er cosinuslikhet mellom sentroidene

In [ ]:
from embedding_similarity import (
    centroid_similarities,
    compute_centroids,
    most_and_least_similar,
)

centroids = compute_centroids(bow.to_numpy(), projection["domain"])
similarities = centroid_similarities(centroids)
most_and_least_similar(similarities).round(3)